In [9]:
#Primero investigamos que campos podrían ser interesantes para buscar con dataformat

!datasets summary gene symbol blaZ --taxon "Staphylococcus aureus"


Error: There are no annotations matching your query at the species level for 'Staphylococcus aureus' (taxid: '1280') please use a subspecies tax name or ID.

Use datasets summary gene symbol <command> --help for detailed help about a command.



In [12]:
#Vamos a obtener los diferentes accession numbers de los genomas de S. aureus, para no tener que ir poniendolos manualmente no por uno y que se nos haga un string larguísimo
#Para ello usaremos datasets summary, no download


from pathlib import Path #Para sacar el archivo de la carpeta del genoma y ponerlo en una nueva que nos interese 
import zipfile #para descomprimir el archivo zip descargado 
import json

output = !datasets summary genome taxon "Staphylococcus aureus" \
--assembly-level complete \
--assembly-source refseq \
--limit 2 \
--as-json-lines | dataformat tsv genome --fields accession,organism-name

print(output)



['New version of client (18.18.0) available at https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/win64/datasets.exe.', 'Assembly Accession\tOrganism Name', 'GCF_000418345.1\tStaphylococcus aureus', 'GCF_000597965.1\tStaphylococcus aureus']


In [6]:
from pathlib import Path #Para sacar el archivo de la carpeta del genoma y ponerlo en una nueva que nos interese 
import zipfile #para descomprimir el archivo zip descargado 
import json

#descargamos 1 genoma de S. aureus
!datasets download genome accession GCF_000013425.1 --include genome

# Crear una nueva carpeta para los archivos descargados
new_folder = Path("genome_files") 
new_folder.mkdir(exist_ok=True) # Esta línea es la que realmente crea la carpeta si no existe "Make dir"
# exist_ok es un parámetro de .mkdir() que acepta True o False. Por defecto es False, lo que significa que si la carpeta ya existe Python dará un error.
# Al ponerlo en True le dices que no dé error y simplemente continúe.

#Descomprimir el zip

zip_path= Path("ncbi_dataset.zip") #ruta del archivo zip descargado
with zipfile.ZipFile(zip_path, 'r') as zip_file: 
    zip_file.extractall("genome_files") #extrae todo en la carpeta definitiva

# Por partes: with... es una estructura de Python que se encarga de cerrar el ZIP automáticamente cuando terminas de usarlo
# zipfile.ZipFile(zip_path, 'r') abre el archivo zip en modo lectura ('r' de read). Para másinfomración leer https://docs.python.org/3.14/library/zipfile.html#zipfile-objects
# zip_file.extractall("dir") extrae todo el contenido del zip abierto en la carpeta que este dentro del paréntesis
# (zip_file es el objeto que representa el archivo zip abierto vaya)

for file in new_folder.rglob("*.fna"):  
    # rglob busca el archivo .fna en todas las subcarpetas
    destino = new_folder / file.name  # nueva ruta de destino
    file.rename(destino) # Rename se refiere a cambiar el nombre del path donde lo mandamos

New version of client (18.18.0) available at https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/win64/datasets.exe.











































































































































































































































































































































































































































































































































































































































































































































































































































































































In [7]:
#Ahora eliminaremos todos los arrchivos que no nos interesen
import shutil #Módulo para eliminar archivos y carpetas

zip_path.unlink() # Es del propio módulo Path, elimina el archivo zip descargado

#Para eliminar todas las otras carpetas hay que usar shutil

for item in new_folder.iterdir(): # iterdir() nos lista todo lo que hay en la carpeta
    if item.is_dir():             # si es una carpeta (directorio) (no el .fna) 
        shutil.rmtree(item)       #Elimina todo su contenido + el propio directorio
    elif not item.suffix == ".fna": #elif = else if. "if-elif-else" structures are used when you have multiple, mutually exclusive conditions.Python evaluates conditions in order. It executes the block for the first true condition and skips the rest.
        item.unlink()             #Elimina todo archivo que no es .fna       
